# BiFAM-Next on Colab (v8): one JOB per session - now with attention images

**Round 2 (jobs 12-16):** other encoders (DenseNet121, ConvNeXt-V2-Tiny), 512 px and up to 40 epochs, to enlarge the image-level ensemble. Jobs 12-15 are independent: run them in any order, on any account. Job 16 = DenseNet121 on unseen patients (two seeds).

**Use this notebook for every job from now on.** New in v8: every job keeps its trained model and saves **attention images** (BiFAM and attention-gate maps at three levels, Grad-CAM, overlay) for about 50 test slices, including the misclassified ones, in `runs/next/<task>/<run>/attention/`.

| JOB | What it runs | Time |
|---|---|---|
| **9** | BiFAM-Next 512 px, seed 33 (three-class ensemble, member 3) | ~50-65 min |
| **10** | BiFAM-Next 512 px, seed 44 (three-class ensemble, member 4) | ~50-65 min |
| 11 | BiFAM-Next on the 7,023-image Kaggle dataset | ~95 min |
| 2-5 | Ablations of BiFAM-Next (two variants per job) | ~75 min each |

(Jobs 1, 6, 7 and 8 are done or not needed.)

**Every session:**
1. **Runtime → Change runtime type → T4 GPU → Save**.
2. In **Step 5**, set `JOB = 9` (next time `JOB = 10`).
3. **Runtime → Run all**. Keep the tab open and the computer awake until you see `[job N] finished`.
4. Tell Claude "done" - the results and attention images are read directly from `MyDrive/BiFAM/runs/next`.


## Step 1 - Check the GPU and connect Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/BiFAM'
os.makedirs(PROJECT, exist_ok=True)
for sub in ['figshare', 'br35h']:          # for datasets you upload yourself
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)
for f in ['bifamnet.py', 'run_local.py']:
    print(f, 'OK' if os.path.exists(f'{PROJECT}/{f}') else '<-- MISSING: upload it to MyDrive/BiFAM')

## Step 2 - Install packages (about 1 minute)

In [ ]:
!pip -q install timm SimpleITK h5py kagglehub

## Step 3 - Kaggle token (only the first time the four-class task needs Br35H)
Br35H is downloaded from Kaggle. If the download asks for a login: on kaggle.com go to **Settings → API → Create New Token**, then upload the downloaded `kaggle.json` to `MyDrive/BiFAM`. This cell copies it into place. You can skip it if no token is needed.

In [ ]:
import shutil
if os.path.exists(f'{PROJECT}/kaggle.json'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy(f'{PROJECT}/kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle token installed')
else:
    print('No kaggle.json in MyDrive/BiFAM (fine unless the Br35H download fails)')

## Step 4 - Restore data prepared in an earlier session (saves 5-15 minutes)

In [ ]:
import glob, subprocess
for z in sorted(glob.glob(f'{PROJECT}/prepared_*.zip')):
    target = '/content/' + os.path.basename(z)[len('prepared_'):-4]
    if not os.path.exists(target):
        print('restoring', target)
        subprocess.run(['unzip', '-q', z, '-d', '/content'], check=True)
print('prepared data on this machine:', [d for d in os.listdir('/content') if d.startswith('data')])

## Step 5 - Choose the job  ← **edit this cell each session**

In [ ]:
JOB = 12       # round 2: 12, 13 = DenseNet121 512 px; 14 = ConvNeXt-V2-Tiny; 15 = DenseNet121 384 px; 16 = DenseNet121 patient level
BATCH = None   # leave None; set 8 only if you get 'CUDA out of memory'


## Step 6 - Run the job (the long step, up to about 100 min)

In [ ]:
import sys, importlib
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
import bifamnet, run_local
importlib.reload(bifamnet); importlib.reload(run_local)

# --- download fix (figshare/Br35H): API download, zip check, or zips you upload to My Drive > BiFAM > figshare / br35h ---
import json as _json, shutil as _shutil, zipfile as _zipfile, urllib.request as _url
from pathlib import Path as _P
_UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

def _fs_ready(d):
    return _P(d).exists() and any(p.stem.isdigit() for p in _P(d).rglob("*.mat"))

def _unpack(dest, *srcs):
    dest = _P(dest); dest.mkdir(parents=True, exist_ok=True)
    roots = [_P(s) for s in srcs if s and _P(s).exists()] + [dest]; done = set()
    while True:
        todo = [z for r in roots for z in r.rglob("*.zip") if z.resolve() not in done]
        if not todo:
            return
        for z in todo:
            done.add(z.resolve())
            if _zipfile.is_zipfile(z):
                print("  unpacking", z.name)
                with _zipfile.ZipFile(z) as f:
                    f.extractall(dest / z.stem if z.parent != dest else dest)
            else:
                print(f"  skipping {z.name}: not a valid zip ({z.stat().st_size} bytes)")

def _fetch(url, path):
    with _url.urlopen(_url.Request(url, headers=_UA), timeout=120) as r, open(path, "wb") as f:
        _shutil.copyfileobj(r, f, 1 << 20)
    if _zipfile.is_zipfile(path):
        return True
    print(f"  {_P(path).name}: the server returned a web page, not a zip"); _P(path).unlink(missing_ok=True)
    return False

def download_figshare(raw, local=f"{PROJECT}/figshare", download=True):
    raw = _P(raw); raw.mkdir(parents=True, exist_ok=True)
    if _fs_ready(raw):
        return True
    _unpack(raw, local)
    if _fs_ready(raw) or not download:
        return _fs_ready(raw)
    try:
        print("[figshare] downloading the four data files (~880 MB) via the figshare API")
        req = _url.Request("https://api.figshare.com/v2/articles/1512427", headers=_UA)
        for f in [f for f in _json.load(_url.urlopen(req, timeout=60))["files"] if f["name"].endswith(".zip")]:
            if not _zipfile.is_zipfile(raw / f["name"]):
                print(f"  {f['name']} ({f['size'] / 2**20:.0f} MB)"); _fetch(f["download_url"], raw / f["name"])
        _unpack(raw)
    except Exception as e:
        print("[figshare] API download failed:", e)
    if not _fs_ready(raw):
        print("[figshare] AUTOMATIC DOWNLOAD FAILED. In your browser open https://doi.org/10.6084/m9.figshare.1512427, "
              "click 'Download all', upload the .zip to My Drive > BiFAM > figshare, then Runtime > Run all again.")
    return _fs_ready(raw)

def download_br35h(raw, local=f"{PROJECT}/br35h", download=True):
    raw = _P(raw); raw.mkdir(parents=True, exist_ok=True)
    ready = lambda: any(d.is_dir() and d.name.lower() == "no" for d in raw.rglob("*"))
    if ready():
        return True
    _unpack(raw, local)
    if ready() or not download:
        return ready()
    try:
        import kagglehub
        _shutil.copytree(kagglehub.dataset_download("ahmedhamada0/brain-tumor-detection"), raw, dirs_exist_ok=True)
    except Exception as e:
        print("[br35h] download failed:", e)
    if not ready():
        print("[br35h] AUTOMATIC DOWNLOAD FAILED. Upload kaggle.json to My Drive > BiFAM (Step 3), or download "
              "https://www.kaggle.com/datasets/ahmedhamada0/brain-tumor-detection in your browser and upload the .zip "
              "to My Drive > BiFAM > br35h, then Runtime > Run all again.")
    return ready()

if not hasattr(bifamnet, "_figshare_ready"):          # older bifamnet.py: use the fixed download functions
    bifamnet.download_figshare, bifamnet.download_br35h = download_figshare, download_br35h
# --- end of download fix ---

# --- fix 2: Br35H duplicate filter (the old one chained similar slices and dropped ~70% of the images) ---
def _thumb(p, size=64):
    a = np.asarray(Image.fromarray(bifamnet.to_gray(Image.open(p))).convert("L").resize((size, size), Image.BILINEAR), np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-6)

def duplicate_clusters(paths, max_hamming=4, max_pixel_diff=0.01):
    n = len(paths); label = np.full(n, -1); first = {}
    for i, p in enumerate(paths):                       # exact copies (identical files)
        h = bifamnet.md5(p)
        if h in first:
            label[i] = label[first[h]]
        else:
            first[h] = i; label[i] = i
    hashes = np.array([bifamnet.phash64(bifamnet.to_gray(Image.open(p))) for p in paths], dtype=np.uint64)
    bits = np.unpackbits(hashes.view(np.uint8).reshape(n, 8), axis=1).astype(np.uint8)
    thumbs, reps = {}, []
    for i in range(n):                                  # near copies: similar hash AND nearly identical pixels, no chaining
        if label[i] != i:
            continue
        match = None
        if reps:
            r = np.array(reps)
            for j in r[(bits[r] != bits[i]).sum(1) <= max_hamming]:
                thumbs.setdefault(j, _thumb(paths[j])); thumbs.setdefault(i, _thumb(paths[i]))
                if np.abs(thumbs[i] - thumbs[j]).mean() <= max_pixel_diff:
                    match = int(j); break
        if match is None:
            reps.append(i)
        else:
            label[i] = label[match]
    return np.array([label[label[i]] for i in range(n)])

import numpy as np
from PIL import Image
bifamnet.duplicate_clusters = duplicate_clusters

# --- fix 3: 'strong' preset for a 1-2 h Colab session (the 'fast' preset stopped too early) ---
run_local.PRESETS['strong'] = dict(seeds=[11], img_size=384, slices_per_case=20, epochs=30, patience=10, batch=16, n_images=40)
bifamnet.DEFAULT_TRAIN['encoder_lr_mult'] = 0.3        # let the pretrained encoder adapt faster (was 0.1)
# --- end of fixes ---


# ================= jobs (ablation, BiFAM-Lite, latency) =================
# ==== BiFAM-Next: every remaining experiment of the manuscript, one JOB per Colab session =========================
# Needs: bifamnet and run_local imported. All jobs use the 'next' protocol: 1 seed (11), 384 px, <= 30 epochs,
# early stopping with patience 10, encoder learning-rate multiplier 0.3. Results: <runs>/next/<task>/<run name>/
import json, sys, time
from pathlib import Path
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

CNX = 'convnextv2_nano.fcmae_ft_in22k_in1k'
run_local.PRESETS['next'] = dict(seeds=[11], img_size=384, slices_per_case=20, epochs=30, patience=10, batch=16, n_images=40)
bifamnet.DEFAULT_TRAIN['encoder_lr_mult'] = 0.3

# --- BiFAM-Next pieces: pyramid pooling (switchable with model cfg 'ppm') and label smoothing ---
if not hasattr(bifamnet, 'PyramidPool'):
    class PyramidPool(nn.Module):
        """Multi-scale context: average-pool the bottleneck to 1x1, 2x2, 3x3 and 6x6, project, upsample, concatenate, fuse."""
        def __init__(self, c, bins=(1, 2, 3, 6)):
            super().__init__()
            self.stages = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(b), nn.Conv2d(c, c // 4, 1, bias=False),
                                                       nn.BatchNorm2d(c // 4), nn.ReLU(inplace=True)) for b in bins])
            self.fuse = nn.Sequential(nn.Conv2d(c + len(bins) * (c // 4), c, 1, bias=False), nn.BatchNorm2d(c),
                                      nn.ReLU(inplace=True))

        def forward(self, x):
            h, w = x.shape[-2:]
            return self.fuse(torch.cat([x] + [F.interpolate(s(x), size=(h, w), mode='bilinear', align_corners=False)
                                              for s in self.stages], 1))

    _Lite = bifamnet.BiFAMLite

    class BiFAMNext(_Lite):
        def __init__(self, **cfg):
            super().__init__(**cfg)
            if self.cfg.get('ppm', 'convnext' in str(self.cfg.get('backbone', ''))):
                self.bottleneck = nn.Sequential(self.bottleneck, PyramidPool(self.cfg['bottleneck_channels']))
    bifamnet.PyramidPool, bifamnet.BiFAMLite = PyramidPool, BiFAMNext

if not hasattr(bifamnet, '_plain_loss'):
    bifamnet._plain_loss = bifamnet.loss_fn
_plain_loss = bifamnet._plain_loss


def smooth_loss(logits, y, num_outputs, eps=0.1):
    if num_outputs == 1:
        return F.binary_cross_entropy_with_logits(logits.squeeze(1), y.float() * (1 - eps) + 0.5 * eps)
    return F.cross_entropy(logits, y.long(), label_smoothing=eps)


def _m(**kw):
    """run_training arguments; every switch goes through the model configuration (works with every bifamnet.py)."""
    return dict(model_over={'backbone': CNX, **kw})


NEXT = _m()                                                                   # full BiFAM-Next
LITE = _m(backbone='densenet121', ppm=False)                                  # BiFAM-Lite (earlier model)
P3 = 'threeclass_patient'
# run name -> (manuscript item, task, run_training arguments, label smoothing)
RUNS = {
    'bifamlite':       ('BiFAM-Next, full model', None, NEXT, True),
    'abl_no_ppm':      ('Ablation: without pyramid pooling', P3, _m(ppm=False), True),
    'abl_no_mask':     ('Ablation: without mask supervision', P3, _m(seg_head=False), True),
    'abl_concat':      ('Ablation: BiFAM -> concatenation', P3, _m(fusion='concat'), True),
    'abl_no_ag':       ('Ablation: attention-gated pathway removed', P3, _m(use_ag=False), True),
    'abl_no_ca':       ('Ablation: channel attention removed', P3, _m(use_ca=False), True),
    'abl_pool':        ('Ablation: transformer head -> global pooling', P3, _m(head='pool'), True),
    'abl_no_ls':       ('Ablation: without label smoothing', P3, NEXT, False),
    'abl_densenet121': ('Ablation: DenseNet121 encoder instead of ConvNeXt-V2', P3, _m(backbone='densenet121', ppm=True), True),
    'lite':            ('BiFAM-Lite (DenseNet121, no PPM, no label smoothing)', None, LITE, False),
    # final image-level model: 512 px, three seeds, averaged with the 384 px seed-11 run (ensemble fixed in advance)
    'hr512_s22':       ('BiFAM-Next 512 px, seed 22 (ensemble member)', None, dict(NEXT, img_size=512, seeds=[22]), True),
    'hr512_s33':       ('BiFAM-Next 512 px, seed 33 (ensemble member)', None, dict(NEXT, img_size=512, seeds=[33]), True),
    'hr512_s44':       ('BiFAM-Next 512 px, seed 44 (ensemble member)', None, dict(NEXT, img_size=512, seeds=[44]), True),
    # round 2 (towards a higher image-level result): other encoders, longer training, members for a larger ensemble
    'dn512_s22':       ('DenseNet121 encoder, 512 px, seed 22', None, dict(_m(backbone='densenet121', ppm=True), img_size=512, seeds=[22], epochs=40), True),
    'dn512_s33':       ('DenseNet121 encoder, 512 px, seed 33', None, dict(_m(backbone='densenet121', ppm=True), img_size=512, seeds=[33], epochs=40), True),
    'cnxtiny_s11':     ('ConvNeXt-V2-Tiny encoder, 384 px, seed 11', None, dict(_m(backbone='convnextv2_tiny.fcmae_ft_in22k_in1k'), seeds=[11], epochs=40), True),
    'dn384_s44':       ('DenseNet121 encoder, 384 px, seed 44', None, dict(_m(backbone='densenet121', ppm=True), seeds=[44], epochs=40), True),
    'dn_s22':          ('DenseNet121 encoder, seed 22 (patient level)', P3, dict(_m(backbone='densenet121', ppm=True), seeds=[22]), True),
    'dn_s33':          ('DenseNet121 encoder, seed 33 (patient level)', P3, dict(_m(backbone='densenet121', ppm=True), seeds=[33]), True),
}
# JOB -> list of (task, run name); each job fits one 1-2 h Colab session on a T4
JOBS = {
    1: [(P3, 'bifamlite')],                                   # ~40 min + 3 min latency: Tables 2, 3, 4, 5 (patient level)
    2: [(P3, 'abl_no_ppm'), (P3, 'abl_no_mask')],             # ~75 min: Table 7
    3: [(P3, 'abl_concat'), (P3, 'abl_no_ag')],               # ~75 min: Table 7
    4: [(P3, 'abl_no_ca'), (P3, 'abl_pool')],                 # ~75 min: Table 7
    5: [(P3, 'abl_no_ls'), (P3, 'abl_densenet121')],          # ~75 min: Table 7
    6: [(P3, 'lite'), ('threeclass_image', 'lite')],          # ~75 min: Table 5 (BiFAM-Lite, three-class)
    7: [('fourclass_image', 'lite')],                         # ~45 min: Table 5 (BiFAM-Lite, four-class)
    8: [('threeclass_image', 'hr512_s22')],                   # ~65 min: image-level ensemble member 1
    9: [('threeclass_image', 'hr512_s33')],                   # ~65 min: image-level ensemble member 2
    10: [('threeclass_image', 'hr512_s44')],                  # ~65 min: image-level ensemble member 3
    12: [('threeclass_image', 'dn512_s22')],                  # ~70 min: DenseNet121, 512 px
    13: [('threeclass_image', 'dn512_s33')],                  # ~70 min: DenseNet121, 512 px
    14: [('threeclass_image', 'cnxtiny_s11')],                # ~60 min: larger ConvNeXt-V2 encoder
    15: [('threeclass_image', 'dn384_s44')],                  # ~45 min: DenseNet121, 384 px
    16: [(P3, 'dn_s22'), (P3, 'dn_s33')],                     # ~70 min: DenseNet121 on unseen patients, two more seeds
}


def measure_latency(out, size=384):
    """Table 2: parameters, GMAC and latency per slice of the three BiFAM networks (GPU, and CPU for BiFAM-Next)."""
    rows = []
    specs = [('BiFAM-Next', 'bifamlite', dict(backbone=CNX)), ('BiFAM-Lite', 'bifamlite', dict(backbone='densenet121', ppm=False)),
             ('BiFAM-Net', 'bifamnet', {})]
    devs = [torch.device('cuda')] if torch.cuda.is_available() else []
    for name, model, cfg in specs:
        for dev in devs + [torch.device('cpu')]:
            if dev.type == 'cpu' and name != 'BiFAM-Next':
                continue
            m = bifamnet.build_model(model, num_outputs=4, img_size=size, pretrained=False, **cfg)
            r = bifamnet.profile(m, dev, size, warmup=20 if dev.type == 'cuda' else 3, runs=200 if dev.type == 'cuda' else 10)
            r = dict(model=name, device=dev.type, **r); rows.append(r); print('[latency]', r)
            del m
    pd.DataFrame(rows).to_csv(out, index=False)


def make_attention(data_root, task, out, device=None, n=40, max_errors=12):
    """Attention maps (BiFAM and attention gate at three levels, Grad-CAM, overlay) for about n test slices spread over
    the classes plus up to max_errors misclassified ones; saved in <run>/attention/ with a contact sheet."""
    out = Path(out)
    if (out / 'attention').exists():
        return
    dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    df = bifamnet.load_index(data_root, bifamnet.TASKS[task][0]).set_index('path', drop=False)
    test = df.loc[bifamnet.load_split(data_root, task)['test']]
    for ck in sorted(out.glob('seed*.pt')):
        seed = int(ck.stem[4:].split('_')[0])
        model, _ = bifamnet.load_checkpoint(ck, dev)
        rows = [bifamnet._pick(test, n, seed)]
        pf = out / f'predictions_seed{seed}.csv'
        if pf.exists():
            pr = pd.read_csv(pf, keep_default_na=False)
            prob = pr[[c for c in pr.columns if c.startswith('prob_')]].values
            wrong = [p for p in pr.path[prob.argmax(1) != pr.label.values] if p in test.index][:max_errors]
            rows.append(test.loc[wrong])
        rows = pd.concat(rows).drop_duplicates('path')
        bifamnet.attention_gallery(data_root, model, rows, bifamnet.TASKS[task][1], out / 'attention', task)
        del model


def run_job(job, settings):
    P = run_local.PRESETS['next']
    data = settings['data'] + (f"_s{P['slices_per_case']}" if P['slices_per_case'] != bifamnet.SLICES_PER_CASE else '')
    runs = Path(settings['runs']) / 'next'
    todo = JOBS[job]
    # prepare the data if this machine does not have it yet (no training here)
    missing = sorted({t for t, _ in todo if t != 'kaggle4_image' and not (Path(data) / 'splits' / f'{t}.json').exists()})
    data_k4 = settings.get('data_k4', settings['data'] + '_k4')
    if any(t == 'kaggle4_image' for t, _ in todo) and not k4_prepare(data_k4, settings.get('raw', 'raw') + '/kaggle4',
                                                                      settings.get('local_k4', 'kaggle4')):
        return
    if missing:
        run_local.SETTINGS.update(settings, backbone=CNX, run_main=False, run_figures=False, run_baselines=False,
                                  run_ablation=False, run_fusion=False, run_analysis=False)
        sys.argv = ['run_local.py', '--tasks', *missing, '--preset', 'next', '--skip-doctor']
        run_local.main()
    import inspect
    _params = inspect.signature(bifamnet.run_training).parameters
    train_over = dict(epochs=P['epochs'], batch=P['batch'], workers=settings.get('workers', 2), tta=1, patience=P['patience'])
    t_start = time.time()
    for task, name in todo:
        item, _, kw, ls = RUNS[name]
        out = runs / task / name
        if (out / 'summary.json').exists():
            print(f'[job {job}] {task}/{name} already finished - skipping'); continue
        print(f'\n[job {job}] ===== {task} / {name}: {item} (label smoothing {"on" if ls else "off"}) =====')
        bifamnet.loss_fn = smooth_loss if ls else _plain_loss
        try:
            opt = {k: v for k, v in dict(keep_checkpoints=True, channels='t1ce').items() if k in _params}   # kept for the attention maps
            kw = dict(kw); size = kw.pop('img_size', P['img_size']); seeds = kw.pop('seeds', P['seeds']); ep = kw.pop('epochs', None)
            tov = dict(train_over, batch=min(train_over['batch'], 12)) if size > 400 else dict(train_over)   # 512 px fits a T4 with batch 12
            if ep:
                tov['epochs'] = ep
            bifamnet.run_training(data_k4 if task == 'kaggle4_image' else data, task, out, model='bifamlite', seeds=seeds, train_over=tov,
                                  img_size=size, device=settings.get('device'), **opt, **kw)
        finally:
            bifamnet.loss_fn = smooth_loss
        try:
            make_attention(data_k4 if task == 'kaggle4_image' else data, task, out, settings.get('device'))
        except Exception:
            import traceback; print('[attention] failed:'); traceback.print_exc()
        print(f'[job {job}] {name} done after {(time.time() - t_start) / 60:.0f} min in this session')
    if job == 1 and not (runs / 'latency.csv').exists():
        measure_latency(runs / 'latency.csv')
    bifamnet.collect_results(runs)
    print('\n================ RESULTS SO FAR (macro F1 / accuracy, %) ================')
    for s in sorted(runs.glob('*/*/summary.json')):
        m = json.load(open(s)).get('mean', {})
        if m.get('macro_f1') is not None:
            print(f"{s.parent.parent.name:20s} {s.parent.name:18s} macro F1 {100 * m['macro_f1']:6.2f}   accuracy {100 * m['accuracy']:6.2f}")
    print(f'\n[job {job}] finished. Send me the results zip (next step).')

# ==== JOB 11: BiFAM-Next on the 7,023-image Brain Tumor MRI Dataset (Nickparvar; figshare + SARTAJ + Br35H) ======
# Official split: Training (5,712) -> train + 10% inner validation, Testing (1,311) -> test. No masks in this dataset,
# so the mask losses are off for every image. Near-copies between the official training and test folders are counted
# (same rule as our Br35H filter) so that the result can also be reported on the test images without a training copy.
import zipfile as _zf
import numpy as np
from PIL import Image
K4_CLASSES = {'glioma': 0, 'meningioma': 1, 'pituitary': 2, 'notumor': 3, 'no_tumor': 3}
bifamnet.TASKS['kaggle4_image'] = ('kaggle4', bifamnet.FOUR_CLASSES, 4, 'image')
_load_index_orig = getattr(bifamnet, '_load_index_orig', bifamnet.load_index)
bifamnet._load_index_orig = _load_index_orig


def _load_index_k4(root, task):
    if task == 'kaggle4':
        return pd.read_csv(Path(root) / 'kaggle4.csv', keep_default_na=False)
    return _load_index_orig(root, task)
bifamnet.load_index = _load_index_k4


def _k4_find(raw):
    """Folder that contains Training/ and Testing/ with the four class folders."""
    for d in [Path(raw)] + [p for p in Path(raw).rglob('*') if p.is_dir()]:
        if (d / 'Training').is_dir() and (d / 'Testing').is_dir():
            return d
    return None


def k4_download(raw, local):
    raw = Path(raw); raw.mkdir(parents=True, exist_ok=True)
    if _k4_find(raw):
        return _k4_find(raw)
    for z in sorted(Path(local).glob('*.zip')) if Path(local).exists() else []:     # zip uploaded to Drive
        if _zf.is_zipfile(z):
            print('[kaggle4] unpacking', z.name); _zf.ZipFile(z).extractall(raw)
    if not _k4_find(raw):
        try:
            import kagglehub, shutil
            print('[kaggle4] downloading masoudnickparvar/brain-tumor-mri-dataset (~150 MB)')
            shutil.copytree(kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset'), raw, dirs_exist_ok=True)
        except Exception as e:
            print('[kaggle4] download failed:', e)
    root = _k4_find(raw)
    if root is None:
        print('[kaggle4] DOWNLOAD FAILED. In your browser open https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset, '
              'click Download, upload the .zip to My Drive > BiFAM > kaggle4, then Runtime > Run all again.')
    return root


def k4_prepare(data, raw, local, seed=0):
    data = Path(data)
    if (data / 'splits' / 'kaggle4_image.json').exists():
        return True
    src = k4_download(raw, local)
    if src is None:
        return False
    rows, n = [], 0
    for part in ('Training', 'Testing'):
        for cdir in sorted((src / part).iterdir()):
            if not cdir.is_dir() or cdir.name.lower() not in K4_CLASSES:
                continue
            for f in sorted(p for p in cdir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')):
                rel = f'kaggle4/{part.lower()}/{n:05d}.png'; n += 1
                (data / rel).parent.mkdir(parents=True, exist_ok=True)
                img = bifamnet.to_uint8(bifamnet.to_gray(Image.open(f)))
                Image.fromarray(bifamnet.resize(img)).save(data / rel)
                rows.append(dict(path=rel, label=K4_CLASSES[cdir.name.lower()], group=f'k4_{n}', source='kaggle4',
                                 mask='', part=part, original=f'{part}/{cdir.name}/{f.name}'))
    df = pd.DataFrame(rows); df.to_csv(data / 'kaggle4.csv', index=False)
    print('[kaggle4] images per part and class:\n', df.groupby(['part', 'label']).size().unstack())
    tr = df[df.part == 'Training']; te = df[df.part == 'Testing']
    rng = np.random.default_rng(seed); val = []
    for _, g in tr.groupby('label'):                                   # stratified 10% inner validation
        val += list(rng.choice(g.path.values, max(1, round(0.1 * len(g))), replace=False))
    val = set(val)
    split = dict(train=[p for p in tr.path if p not in val], val=sorted(val), test=list(te.path))
    (data / 'splits').mkdir(parents=True, exist_ok=True)
    json.dump(split, open(data / 'splits' / 'kaggle4_image.json', 'w'))
    # near-copies across the official split (exact MD5, or pHash <= 4 bits and mean thumbnail difference <= 0.01)
    paths = [data / p for p in df.path]
    lab = bifamnet.duplicate_clusters(paths)
    df['cluster'] = lab
    train_clusters = set(df.cluster[df.part == 'Training'])
    leak = df[(df.part == 'Testing') & df.cluster.isin(train_clusters)]
    json.dump(dict(n_test=int(len(te)), test_with_training_copy=list(leak.path),
                   n_leak=int(len(leak)), clusters_removed=int(len(df) - df.cluster.nunique())),
              open(data / 'kaggle4_duplicates.json', 'w'), indent=1)
    print(f'[kaggle4] {len(leak)} of {len(te)} test images have a near-copy in the training folder')
    return True


RUNS['k4_next'] = ('BiFAM-Next on the 7,023-image Brain Tumor MRI Dataset (no masks)', None, _m(seg_head=False), True)
JOBS[11] = [('kaggle4_image', 'k4_next')]

run_local.SETTINGS.update(raw='/content/raw', data='/content/data_v2', runs=f'{PROJECT}/runs', brats=None,
                          device='cuda:0', workers=2)
if BATCH:
    run_local.PRESETS['next']['batch'] = BATCH
import os as _os; _os.makedirs(f'{PROJECT}/kaggle4', exist_ok=True)
run_job(JOB, dict(raw='/content/raw', data='/content/data_v2', runs=f'{PROJECT}/runs', device='cuda:0', workers=2,
                  data_k4='/content/data_k4', local_k4=f'{PROJECT}/kaggle4'))


## Step 7 - Save the prepared data to Drive (only matters if data were prepared in this session)

In [ ]:
for d in sorted(glob.glob('/content/data*')):
    z = f'{PROJECT}/prepared_{os.path.basename(d)}.zip'
    newest = max((os.path.getmtime(f) for f in glob.glob(d + '/splits/*.json')), default=0)
    if not os.path.exists(z) or newest > os.path.getmtime(z):      # new or re-prepared data
        print('saving', z)
        if os.path.exists(z):
            os.remove(z)
        subprocess.run(f'cd /content && zip -qr "{z}" "{os.path.basename(d)}"', shell=True, check=True)
print('done')

## Step 8 - Download the results zip and send it to Claude

In [ ]:
from google.colab import files
subprocess.run(f'cd "{PROJECT}/runs" && zip -qr /content/results_job{JOB}.zip next -x "*.pt" -x "next/figures/*" -x "*/features/*"',
               shell=True, check=True)
files.download(f'/content/results_job{JOB}.zip')
